#Load the necessary library


In [19]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


# Question 1 : Load both the file

In [20]:
ledger_df = pd.read_csv('ledger.csv')
gateway_df = pd.read_csv('gateway.csv')


print("\nLedger Shape :", ledger_df.shape)
print("Gateway Shape:", gateway_df.shape)

print("\nLedger Columns:")
print(ledger_df.columns.tolist())

print("\nGateway Columns:")
print(gateway_df.columns.tolist())


Ledger Shape : (10, 6)
Gateway Shape: (9, 6)

Ledger Columns:
['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd', 'status', 'payment_method']

Gateway Columns:
['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd', 'status', 'payment_method']


In [21]:
ledger_df = ledger_df.rename(columns={
    'transaction_date': 'ledger_date',
    'amount_usd': 'ledger_amount',
    'status': 'ledger_status'
})

gateway_df = gateway_df.rename(columns={
    'transaction_date': 'gateway_date',
    'amount_usd': 'gateway_amount',
    'status': 'gateway_status'
})

# Question 2 : Check Duplicate AND Nulls

In [22]:
print("\n DUPLICATE CHECK")

ledger_duplicates = ledger_df.duplicated().sum()
gateway_duplicates = gateway_df.duplicated().sum()

print(f"Ledger Duplicate Rows : {ledger_duplicates}")
print(f"Gateway Duplicate Rows: {gateway_duplicates}")


 DUPLICATE CHECK
Ledger Duplicate Rows : 0
Gateway Duplicate Rows: 0


# Question 2 (b) Null Value Check



In [23]:
print("\n================ NULL VALUE CHECK ================")

ledger_nulls = ledger_df.isnull().sum()
gateway_nulls = gateway_df.isnull().sum()

print("\nLedger Null Values:")
print(ledger_nulls)

print("\nGateway Null Values:")
print(gateway_nulls)


================ NULL VALUE CHECK ================

Ledger Null Values:
transaction_id    0
ledger_date       0
merchant_id       0
ledger_amount     0
ledger_status     0
payment_method    0
dtype: int64

Gateway Null Values:
transaction_id    0
gateway_date      0
merchant_id       0
gateway_amount    0
gateway_status    0
payment_method    0
dtype: int64


# Question 3 : identify records missing in gateway


In [24]:
ledger_ids = set(ledger_df['transaction_id'])
gateway_ids = set(gateway_df['transaction_id'])

missing_in_gateway_ids = ledger_ids - gateway_ids

missing_in_gateway = ledger_df[
    ledger_df['transaction_id'].isin(missing_in_gateway_ids)
].copy()

missing_in_gateway['issue_type'] = 'Missing in Gateway'

print("\n================ MISSING IN GATEWAY ================")
print(f"Total Records Missing in Gateway: {len(missing_in_gateway)}")

display(missing_in_gateway.head())



================ MISSING IN GATEWAY ================
Total Records Missing in Gateway: 2


,transaction_id,ledger_date,merchant_id,ledger_amount,ledger_status,payment_method,issue_type
3,R004,2026-03-02,M003,2100.0,success,Card,Missing in Gateway
9,R010,2026-03-05,M004,2500.0,success,Wallet,Missing in Gateway


# Question 4 : identify records missing in ledger

In [25]:
missing_in_ledger_ids = gateway_ids - ledger_ids

missing_in_ledger = gateway_df[
    gateway_df['transaction_id'].isin(missing_in_ledger_ids)
].copy()

missing_in_ledger['issue_type'] = 'Missing in Ledger'

print("\n MISSING IN LEDGER ")
print(f"Total Records Missing in Ledger: {len(missing_in_ledger)}")

display(missing_in_ledger.head())


 MISSING IN LEDGER 
Total Records Missing in Ledger: 1


,transaction_id,gateway_date,merchant_id,gateway_amount,gateway_status,payment_method,issue_type
8,R011,2026-03-05,M003,1800.0,success,Card,Missing in Ledger


# Question 5 : identify amount mismatches

In [26]:
common_ids = ledger_ids.intersection(gateway_ids)

ledger_common = ledger_df[
    ledger_df['transaction_id'].isin(common_ids)
]

gateway_common = gateway_df[
    gateway_df['transaction_id'].isin(common_ids)
]

merged_amount = pd.merge(
    ledger_common,
    gateway_common,
    on='transaction_id',
    suffixes=('_ledger', '_gateway')
)


amount_mismatch = merged_amount[
    merged_amount['ledger_amount'] != merged_amount['gateway_amount']
].copy()

amount_mismatch['amount_difference'] = (
    amount_mismatch['ledger_amount']
    - amount_mismatch['gateway_amount']
)

amount_mismatch['issue_type'] = 'Amount Mismatch'

print("\n================ AMOUNT MISMATCHES ================")
print(f"Total Amount Mismatches: {len(amount_mismatch)}")

display(
    amount_mismatch[
        [
            'transaction_id',
            'ledger_amount',
            'gateway_amount',
            'amount_difference',
            'issue_type'
        ]
    ].head()
)



================ AMOUNT MISMATCHES ================
Total Amount Mismatches: 2


,transaction_id,ledger_amount,gateway_amount,amount_difference,issue_type
1,R002,850.0,900.0,-50.0,Amount Mismatch
6,R008,640.0,600.0,40.0,Amount Mismatch


# Question 6  identify status mismatches



In [27]:
status_mismatch = merged_amount[
    merged_amount['ledger_status'] != merged_amount['gateway_status']
].copy()

status_mismatch['issue_type'] = 'Status Mismatch'

print("\n================ STATUS MISMATCHES ================")
print(f"Total Status Mismatches: {len(status_mismatch)}")

display(
    status_mismatch[
        [
            'transaction_id',
            'ledger_status',
            'gateway_status',
            'issue_type'
        ]
    ].head()
)


================ STATUS MISMATCHES ================
Total Status Mismatches: 1


,transaction_id,ledger_status,gateway_status,issue_type
3,R005,success,failed,Status Mismatch


# Question 6 : build a final reconciliation report

In [28]:
report_missing_gateway = missing_in_gateway[
    ['transaction_id', 'issue_type']
].copy()

# Missing in Ledger Report
report_missing_ledger = missing_in_ledger[
    ['transaction_id', 'issue_type']
].copy()

# Amount Mismatch Report
report_amount = amount_mismatch[
    [
        'transaction_id',
        'ledger_amount',
        'gateway_amount',
        'amount_difference',
        'issue_type'
    ]
].copy()

# Status Mismatch Report
report_status = status_mismatch[
    [
        'transaction_id',
        'ledger_status',
        'gateway_status',
        'issue_type'
    ]
].copy()

# Combine all reports
final_report = pd.concat(
    [
        report_missing_gateway,
        report_missing_ledger,
        report_amount,
        report_status
    ],
    ignore_index=True
)

# Severity Mapping
severity_map = {
    'Missing in Gateway': 'High',
    'Missing in Ledger': 'High',
    'Amount Mismatch': 'Medium',
    'Status Mismatch': 'Medium'
}

final_report['severity'] = final_report['issue_type'].map(severity_map)

print("\n================ FINAL RECONCILIATION REPORT ================")

print(f"Total Issues Found: {len(final_report)}")

display(final_report.head())




================ FINAL RECONCILIATION REPORT ================
Total Issues Found: 6


,transaction_id,issue_type,ledger_amount,gateway_amount,amount_difference,ledger_status,gateway_status,severity
0,R004,Missing in Gateway,NaN,NaN,NaN,NaN,NaN,High
1,R010,Missing in Gateway,NaN,NaN,NaN,NaN,NaN,High
2,R011,Missing in Ledger,NaN,NaN,NaN,NaN,NaN,High
3,R002,Amount Mismatch,850.0,900.0,-50.0,NaN,NaN,Medium
4,R008,Amount Mismatch,640.0,600.0,40.0,NaN,NaN,Medium


# Question 8 : generate summary metrics and CSV file generation for all the summary

In [29]:
final_report.to_csv('final_reconciliation_report.csv', index=False)
print("final_reconciliation_report.csv")



final_reconciliation_report.csv


In [30]:
summary_metrics = {
    'ledger_total_records': len(ledger_df),
    'gateway_total_records': len(gateway_df),

    'ledger_duplicates': ledger_duplicates,
    'gateway_duplicates': gateway_duplicates,

    'ledger_total_nulls': ledger_df.isnull().sum().sum(),
    'gateway_total_nulls': gateway_df.isnull().sum().sum(),

    'missing_in_gateway': len(missing_in_gateway),
    'missing_in_ledger': len(missing_in_ledger),

    'amount_mismatches': len(amount_mismatch),
    'status_mismatches': len(status_mismatch),

    'total_issues': len(final_report)
}

summary_df = pd.DataFrame(summary_metrics.items(), columns=['Metric', 'Value'])

print("SUMMARY METRICS")

display(summary_df)


summary_df.to_csv('summary_metrics.csv', index=False)

print("summary_metrics.csv")

SUMMARY METRICS


,Metric,Value
0,ledger_total_records,10
1,gateway_total_records,9
2,ledger_duplicates,0
3,gateway_duplicates,0
4,ledger_total_nulls,0
5,gateway_total_nulls,0
6,missing_in_gateway,2
7,missing_in_ledger,1
8,amount_mismatches,2
9,status_mismatches,1


summary_metrics.csv


In [31]:
import json
missing_in_gateway.to_csv('missing_in_gateway.csv',index=False)


missing_in_ledger.to_csv('missing_in_ledger.csv',index=False)


amount_mismatch.to_csv('amount_mismatches.csv',index=False)


status_mismatch.to_csv('status_mismatches.csv',index=False)


final_report.to_csv('reconciliation_report.csv',index=False)






summary_metrics = {"ledger_total_records": int(len(ledger_df)),
    "gateway_total_records": int(len(gateway_df)),

    "ledger_duplicates": int(ledger_duplicates),
    "gateway_duplicates": int(gateway_duplicates),

    "ledger_null_values": int(ledger_df.isnull().sum().sum()),
    "gateway_null_values": int(gateway_df.isnull().sum().sum()),

    "missing_in_gateway": int(len(missing_in_gateway)),
    "missing_in_ledger": int(len(missing_in_ledger)),

    "amount_mismatches": int(len(amount_mismatch)),
    "status_mismatches": int(len(status_mismatch)),

    "total_issues": int(len(final_report))
}

with open('summary_metrics.json', 'w') as f:
    json.dump(summary_metrics, f, indent=4)

import os



for file in os.listdir():
    if file.endswith('.csv') or file.endswith('.json'):
        print(file)

amount_mismatches.csv
reconciliation_report.csv
summary_metrics.json
missing_in_gateway.csv
summary_metrics.csv
status_mismatches.csv
gateway.csv
missing_in_ledger.csv
ledger.csv
final_reconciliation_report.csv
